In [ ]:
import pandas as pd
import numpy as np

from math import ceil

#import matplotlib.pyplot as plt
import sys, os, time, random, re, csv, json, argparse, torch #,copy , indexer, evaluate

from datetime import datetime
from datasets import Dataset, Value, concatenate_datasets 
from sklearn.metrics import mean_squared_error, f1_score, accuracy_score, precision_score, recall_score, classification_report
from scipy.special import expit

from torch.nn import functional as F#, BCEWithLogitsLoss
from torch.utils.data import WeightedRandomSampler

from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, 
                          BertForSequenceClassification, BertModel, EarlyStoppingCallback, AdamW,
                          PreTrainedModel, Trainer, TrainingArguments, get_scheduler)

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object #, text_cleansing

In [ ]:
# TEST STRATIFIED
test_trat = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test_strat.csv", sep='|')
test_trat

In [ ]:
def get_dataset(table):
    # --- ### VALUENET    
    if table == "valueNET":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv",sep=',')

        value_train_set = np.loadtxt("/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv", delimiter=',', dtype=int)

    # --- ### VALUEARG
    elif table == "valueARG":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv",sep=',')

        value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv", delimiter=',', dtype=int)

    # --- ### All (valueNET+valueARG)
    elif table == "valueALL":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv",sep=',')
    
        value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)

    return value_set, value_train_set, value_val_set, value_test_set

def get_dataset(table):
    # --- ### VALUENET    
    if table == "valueNET":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv",sep=',')

        value_train_set = np.loadtxt("/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv", delimiter=',', dtype=int)

    # --- ### VALUEARG
    elif table == "valueARG":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv",sep=',')

        value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv", delimiter=',', dtype=int)

    # --- ### All (valueNET+valueARG)
    elif table == "valueALL":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv",sep=',')
    
        value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)

    return value_set, value_train_set, value_val_set, value_test_set

def cast_dataset_to_hf(dataset, split_name, abs_label=True):
    """
    Convert custom dataset to HuggingFace dataset.
    """
    dataset = dataset.copy()  # avoid SettingWithCopyWarning

    if abs_label:
        dataset['label'] = dataset['label'].apply(lambda x: abs(x))

    dataset_dict = {
        'id': dataset['uid'].astype(str).tolist(),
        'text': dataset['scenario'].astype(str).tolist(),
        'orig_label': dataset['label'].astype(int).tolist(),  # ensure label is int
        'value': dataset['value'].astype(str).tolist(),       # make sure this is str
    }

    hf_dataset = Dataset.from_dict(dataset_dict)  # build HuggingFace Dataset
    return hf_dataset


def hf_dataset_tokenize(hf_dataset, tokenizer, soft_target_type='int', input_concat=True):
    base_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer # ROBERTA /DEBERTA

    def tokenize_batch(batch):
        if input_concat:
            # normaliza a minúsculas para que coincida con <achievement> (no <ACHIEVEMENT>)
            values = [str(v).lower() for v in batch["value"]]
            texts  = batch["text"]
            batched_inputs = [f"<{values[i]}> {base_tok.sep_token} {texts[i]}" for i in range(len(texts))]  # ROBERTA /DEBERTA
        else:
            batched_inputs = batch["text"]

        enc = base_tok(
            batched_inputs,
            truncation=True,
            padding="max_length",
            max_length=256
        )
        enc["labels"] = batch["orig_label"]
        enc["value"] = batch["value"]
        return enc

    remove_cols = [c for c in hf_dataset.column_names if c not in ["value"]]
    tokenized_dataset = hf_dataset.map(tokenize_batch, batched=True, remove_columns=remove_cols)
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

    target_type = Value(dtype='float32', id=None) if soft_target_type == 'float' else Value(dtype='int64', id=None)
    tokenized_dataset = tokenized_dataset.cast_column('labels', target_type)
    return tokenized_dataset


class ValueTokenizer():
    def __init__(self, model_name, input_concat=False, label_type="copy", label2id=None):
        self.model_name: str = model_name
        #print(f"[DEBUG] ValueTokenizer __init__, model_name = {self.model_name!r}")  # ADD THIS LINE
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, use_fast=True)
        #self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, use_fast=True, revision="main")
        self.input_concat = input_concat
        self.label_type = label_type
        self.label2id = label2id

    def nhot_labels(self, batch_labels, num_classes=10):
        """
        batch_labels contains list of indices, convert to n-hot encoded matrix.
        """
        batch_size = len(batch_labels)
        labels = np.zeros((batch_size, num_classes), dtype=np.float64)
        for i, ex in enumerate(batch_labels):
            for v in ex:
                labels[i, v] = float(1.0)
        return labels

    def tokenize(self, examples):
        # Gather input text
        if self.input_concat:
            batch_size = len(examples['text'])
            batched_inputs = [f"<{examples['value'][i].lower()}> " + f"{self.tokenizer.sep_token} " + examples['text'][i] for i in range(batch_size)]
        else:
            batched_inputs = examples['text']

        # Tokenize
        samples = self.tokenizer(batched_inputs, truncation=True, padding='max_length', max_length=256)

        # Gather target labels
        if self.label_type == "cast_float":
            samples["labels"] = [float(x) for x in examples["orig_label"]]
        elif self.label_type == "cast_nhot":
            samples['labels'] = self.nhot_labels(examples['labels'], num_classes=10)
        elif self.label_type == "cast_nhot_schwartz":
            batch_labels = []
            for multi_labels in examples['schwartz_labels']:
                batch_labels.append([self.label2id[x] for x in multi_labels])
            samples['labels'] = self.nhot_labels(batch_labels, num_classes=len(self.label2id))
        elif self.label_type == "copy":
            samples['labels'] = examples['labels']
        return samples
    
    # make class "callable"
    def __call__(self, examples):
        return self.tokenize(examples)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = expit(predictions)
    preds_binary = (probs > 0.5).astype(int)

    rmse = np.sqrt(mean_squared_error(labels, probs))
    f1 = f1_score(labels, preds_binary)
    precision = precision_score(labels, preds_binary)
    recall = recall_score(labels, preds_binary)
    accuracy = accuracy_score(labels, preds_binary)

    report = classification_report(labels, preds_binary, target_names=["Negative", "Positive"])
    print("=== Classification Report ===")
    print(report)

    with open("eval_classification_report.txt", "w") as f:
        f.write(report)

    return {
        "rmse": rmse,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy
    }


class FocalLossTrainer(Trainer):
    def __init__(self, train_sampler=None, gamma=1.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        self.train_sampler = train_sampler

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs["labels"].float()
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)
        probs = torch.sigmoid(logits)
        bce = F.binary_cross_entropy(probs, labels, reduction='none')
        pt = torch.exp(-bce)
        focal_loss = ((1 - pt) ** self.gamma * bce).mean()
        return (focal_loss, outputs) if return_outputs else focal_loss
    
    def get_train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=self.train_sampler,
            collate_fn=self.data_collator,
            drop_last=False,
            num_workers=0,
        )

def model_init(checkpoint, tokenizer):
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)
    model.resize_token_embeddings(len(tokenizer))  # tokenizer HF real
    return model

def dump_predictions(args, dataset, results, output_file):
    """
    Dump model predictions to a file.
    """
    with open(output_file, 'w') as f:
        result_dump = {
            'timestamp': str(datetime.now()),
            'cli_args': vars(args),
            'preds': results.predictions.tolist(),
            'true': dataset['labels'].tolist(),
            'metrics': results.metrics,
        }
        json.dump(result_dump, f, indent=2)


def evaluate_model(checkpoint, dataset, value_tokenizer):#, args):
    # Load model from checkpoint
    model = model_init(checkpoint, value_tokenizer)
    
    # Define evaluation args
    test_args = TrainingArguments(
        output_dir=".",
        do_train=False,
        do_eval=False,
        do_predict=True,
        per_device_eval_batch_size=32,
        seed=0
    )
    
    # Initialize trainer
    base_tok = value_tokenizer.tokenizer if hasattr(value_tokenizer, "tokenizer") else value_tokenizer
    
    trainer = Trainer(
        model,
        test_args,
        tokenizer=base_tok,
    )

    # Run prediction
    predictions = trainer.predict(dataset)
    
    # Convert logits → probabilities
    logits = predictions.predictions.squeeze()
    probs = expit(logits)

    # Threshold at 0.5
    preds_binary = (probs > 0.5).astype(int)
    
    safe_model_name = "roberta-base".replace("/", "_")
    #dump_predictions(args, dataset, predictions, f"/Proyecto/Value-disagreement/Python/Results/{safe_model_name}_{args.table}_seed{args.seed}.json")
    
    f1 = f1_score(y_true=predictions.label_ids, y_pred=preds_binary, average='macro')
    accuracy = accuracy_score(y_true=predictions.label_ids, y_pred=preds_binary)
    rmse = np.sqrt(mean_squared_error(predictions.label_ids, probs))
    
    # F1 per value
    value_list = dataset['value']  # list of value names per example
    value_names = sorted(set(value_list))

    print("\n=== PER-VALUE F1 ===")
    per_value_f1_dict = {}
    for v in value_names:
        # Mask for this value
        mask = np.array(value_list) == v
        # Compute F1 for this value
        f1_v = f1_score(np.array(predictions.label_ids)[mask], preds_binary[mask])
        per_value_f1_dict[v] = f1_v
        print(f"{v:>20}: F1 = {f1_v:.3f}")

    # Optional: save per-value F1 to file
 
    per_value_f1_file = f"/Proyecto/Value-disagreement/Python/Results/Per Value/{safe_model_name}_{'valueALL'}_VAL_seed{0}_per_value_f1.json"
    with open(per_value_f1_file, "w") as f_out:
        json.dump(per_value_f1_dict, f_out, indent=2)
        print(f"\nPer-value F1 saved to {per_value_f1_file}")

    print(f"F1 score: {f1:.3f}")
    print(f"Accuracy: {accuracy:.3f}")
    
    return f1, accuracy, rmse


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# ==== BEGIN NORMAL PIPELINE ====      

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_file = f"Results/results_valueALL_strat_bert-base-uncased_{timestamp}.csv"
os.makedirs(os.path.dirname(csv_file), exist_ok=True)

value_set, value_train_set, value_val_set, value_test_set = get_dataset("valueALL")
print(f"Sizes of sets: total: {len(value_set)}, train: {len(value_train_set)}, val: {len(value_val_set)}, test: {len(value_test_set)}")

train_subset = value_set.iloc[value_train_set]

# Compute class weights
label_counts = train_subset["label"].value_counts()
weight_ratio = label_counts[0] / label_counts[1]  # usually >> 1
weights = train_subset["label"].map({0: 1.0, 1: weight_ratio}).values

# Create the sampler
train_sampler = WeightedRandomSampler(weights=weights,
                                      num_samples=len(weights),
                                      replacement=True
                                      )

value_tokenizer = AutoTokenizer.from_pretrained("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed0")

"""value_tokenizer = ValueTokenizer('bert-base-uncased',
                                 input_concat=True,
                                 label_type="cast_float"
                                 )

# Add special value tokens (10 values)
num_added_tokens = value_tokenizer.tokenizer.add_special_tokens(
    {"additional_special_tokens": [f"<{x}>" for x in Dict_Object.ValueConstants.SCHWARTZ_VALUES]})
print(f"Added {num_added_tokens} special value tokens")"""


# Preparing DataSets:::::::::::::::::::::::::::::::::::::::
#train_hf = cast_dataset_to_hf(value_set.filter(value_train_set.astype(np.int64), axis=0), "train", abs_label=True)
#val_hf = cast_dataset_to_hf(value_set.filter(value_val_set.astype(np.int64), axis=0), "val", abs_label=True)
#test_hf = cast_dataset_to_hf(value_set.filter(value_test_set.astype(np.int64), axis=0), "test", abs_label=True)

train_hf = cast_dataset_to_hf(value_set.iloc[value_train_set], "train", abs_label=True)
val_hf = cast_dataset_to_hf(value_set.iloc[value_val_set], "val", abs_label=True)
#test_hf = cast_dataset_to_hf(value_set.iloc[value_test_set], "test", abs_label=True)
test_hf = cast_dataset_to_hf(test_trat, "test", abs_label=True)

# Tokenize data
tokenized_dataset_train = hf_dataset_tokenize(train_hf, value_tokenizer, soft_target_type='float')
tokenized_dataset_val = hf_dataset_tokenize(val_hf, value_tokenizer, soft_target_type='float')
tokenized_dataset_test = hf_dataset_tokenize(test_hf, value_tokenizer, soft_target_type='float')

In [ ]:
test_tok = "<power>"
tok_id = value_tokenizer.convert_tokens_to_ids(test_tok)
unk_id = value_tokenizer.unk_token_id
print(f"{test_tok} -> {tok_id} | <unk> -> {unk_id} | is_unk? {tok_id == unk_id}")

In [ ]:
csv_header = ["run_id", "seed", "f1", "accuracy", "rmse", "train_time_sec", "table", "model", "timestamp"]

with open(csv_file, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(csv_header)

    #f1, accuracy, rmse = evaluate_model(best_model_path, tokenized_dataset_test, value_tokenizer, args)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed0/checkpoint-13069", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed1/checkpoint-11617", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed2/checkpoint-8712", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed3/checkpoint-10164", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed4/checkpoint-5808", tokenized_dataset_val, value_tokenizer)

    f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed0/checkpoint-11617", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed1/checkpoint-11617", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed2/checkpoint-13069", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed3/checkpoint-14520", tokenized_dataset_val, value_tokenizer)
    #f1, accuracy, rmse = evaluate_model("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed4/checkpoint-13069", tokenized_dataset_val, value_tokenizer)
   
    # LOG RESULTS
    #writer.writerow([0, 0, f1, accuracy, rmse, round(25500, 2), 'valueALL', 'microsoft_deberta-v3-base', timestamp])
    #csvfile.flush()

    print(f">>> Run {0} DONE: F1={f1}, Acc={accuracy}, Time={train_time:.1f}s")
    
    print(f"\n=== ALL RUNS DONE! Results saved to {csv_file} ===")